# Lastfm y Acousticbrainz

In [2]:
"""
=============================================================================
  ProyectoFinal23 — Pipeline de Enriquecimiento Musical
  Fuente: Last.fm (lastfm_dataset.csv) + AcousticBrainz API
=============================================================================

  Columnas finales objetivo:
    name, artist, bpm, key, scale,
    danceability, mood_happy, genre_ab,
    country, playcount, listeners

  Dataset de entrada:
    lastfm_dataset.csv — 59.999 tracks, ya tiene columna 'mbid'

  APIs utilizadas (sin API key):
    AcousticBrainz low-level  → https://acousticbrainz.org/{mbid}/low-level
    AcousticBrainz high-level → https://acousticbrainz.org/{mbid}/high-level

  Dependencias:
    pip install requests pandas

=============================================================================
"""

import os
import time
import logging
from datetime import datetime

import pandas as pd
import requests


# =============================================================================
#  CONFIGURACIÓN — ajusta estos valores según tus necesidades
# =============================================================================

CSV_PATH   = "src/lastfm_dataset.csv"   # ruta al dataset de entrada
OUTPUT_DIR = "."                     # carpeta donde guardar los resultados

# Cuántos tracks enriquecer:
#   None  → todos los que tienen MBID (~53.000, unas 15-20 horas)
#   2000  → muestra rápida para prueba (~45-60 min)  ← recomendado para empezar
SAMPLE_SIZE = 2000

SLEEP_SECONDS      = 0.3   # pausa entre llamadas a la API (throttling)
MAX_RETRIES        = 3     # reintentos si la API falla
LOG_EVERY_N        = 100   # mostrar progreso cada N tracks
CHECKPOINT_EVERY_N = 500   # guardar CSV parcial cada N tracks


# =============================================================================
#  LOGGING — muestra mensajes en pantalla y los guarda en pipeline.log
# =============================================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("pipeline.log", encoding="utf-8"),
    ],
)
log = logging.getLogger(__name__)


# =============================================================================
#  FUNCIÓN AUXILIAR: petición HTTP segura con reintentos
# =============================================================================

def _get(url):
    """
    Hace una petición GET a una URL.

    - Si la respuesta es 200 → devuelve el objeto Response.
    - Si la respuesta es 404 → devuelve None (el track no existe en AB).
    - Si hay error de servidor (5xx) o rate limit (429) → reintenta
      hasta MAX_RETRIES veces esperando cada vez más (backoff exponencial).
    - Si hay error de red → reintenta también.

    Siempre devuelve Response o None. Nunca lanza una excepción.
    """
    headers = {"User-Agent": "ProyectoFinal23/1.0 (bootcamp-project@example.com)"}

    for intento in range(1, MAX_RETRIES + 1):
        try:
            respuesta = requests.get(url, headers=headers, timeout=10)

            if respuesta.status_code == 200:
                return respuesta                      # éxito

            elif respuesta.status_code == 404:
                return None                           # track no existe en AB

            elif respuesta.status_code in (429, 500, 502, 503):
                espera = 2 ** intento                 # 2, 4, 8 segundos
                log.warning(f"HTTP {respuesta.status_code} — esperando {espera}s (intento {intento}/{MAX_RETRIES})")
                time.sleep(espera)

            else:
                return None                           # otro error no recuperable

        except requests.exceptions.Timeout:
            log.warning(f"Timeout en intento {intento}/{MAX_RETRIES}")
            time.sleep(2 ** intento)

        except requests.exceptions.ConnectionError:
            log.warning(f"Sin conexión en intento {intento}/{MAX_RETRIES}")
            time.sleep(2 ** intento)

        except Exception as error:
            log.error(f"Error inesperado: {error}")
            return None

    return None   # agotados todos los reintentos


# =============================================================================
#  FUNCIÓN 1: obtener datos LOW-LEVEL de AcousticBrainz
# =============================================================================

def fetch_low_level(mbid):
    """
    Consulta AcousticBrainz low-level para un MBID.

    Devuelve un diccionario con:
      - bpm   (float): pulsaciones por minuto
      - key   (str):   tonalidad, ej: "C", "G#"
      - scale (str):   "major" o "minor"

    Si no hay datos → devuelve el mismo diccionario pero con None en los valores.
    """
    resultado = {"bpm": None, "key": None, "scale": None}

    # Validar que el MBID no está vacío
    if not mbid or pd.isna(mbid):
        return resultado

    url = f"https://acousticbrainz.org/{mbid}/low-level"
    respuesta = _get(url)

    if respuesta is None:
        return resultado

    try:
        datos = respuesta.json()
        resultado["bpm"]   = datos.get("rhythm", {}).get("bpm")
        resultado["key"]   = datos.get("tonal",  {}).get("key_key")
        resultado["scale"] = datos.get("tonal",  {}).get("key_scale")
    except Exception as error:
        log.debug(f"Error parseando low-level (mbid={mbid}): {error}")

    return resultado


# =============================================================================
#  FUNCIÓN 2: obtener datos HIGH-LEVEL de AcousticBrainz
# =============================================================================

def fetch_high_level(mbid):
    """
    Consulta AcousticBrainz high-level para un MBID.

    Devuelve un diccionario con:
      - danceability (float 0-1): probabilidad de ser bailable
      - mood_happy   (float 0-1): probabilidad de mood alegre
      - genre_ab     (str):       género según clasificador Dortmund
                                  ej: "pop", "rock", "electronic"

    Si no hay datos → devuelve el mismo diccionario pero con None en los valores.
    """
    resultado = {"danceability": None, "mood_happy": None, "genre_ab": None}

    if not mbid or pd.isna(mbid):
        return resultado

    url = f"https://acousticbrainz.org/{mbid}/high-level"
    respuesta = _get(url)

    if respuesta is None:
        return resultado

    try:
        datos = respuesta.json()
        hl = datos.get("highlevel", {})

        resultado["danceability"] = hl.get("danceability", {}).get("all", {}).get("danceable")
        resultado["mood_happy"]   = hl.get("mood_happy",   {}).get("all", {}).get("happy")
        resultado["genre_ab"]     = hl.get("genre_dortmund", {}).get("value")
    except Exception as error:
        log.debug(f"Error parseando high-level (mbid={mbid}): {error}")

    return resultado


# =============================================================================
#  FUNCIÓN 3: pipeline completo de enriquecimiento
# =============================================================================

def enrich_dataset_with_acousticbrainz(df, sample_size=None):
    """
    Enriquece el dataset de Last.fm con datos de AcousticBrainz.

    Parámetros:
      df          → DataFrame con columna 'mbid'
      sample_size → número de tracks a procesar (None = todos)

    Devuelve:
      df_final → DataFrame con las 11 columnas objetivo:
                 name, artist, bpm, key, scale, danceability,
                 mood_happy, genre_ab, country, playcount, listeners
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")

    # --- Paso 1: separar tracks con y sin MBID ---
    df_con_mbid    = df.dropna(subset=["mbid"]).copy()
    df_sin_mbid    = df[df["mbid"].isna()].copy()

    log.info("=" * 55)
    log.info("  INICIO DEL PIPELINE DE ENRIQUECIMIENTO")
    log.info("=" * 55)
    log.info(f"  Total tracks en el dataset : {len(df):,}")
    log.info(f"  Tracks con MBID válido     : {len(df_con_mbid):,}")
    log.info(f"  Tracks sin MBID (→ NaN)    : {len(df_sin_mbid):,}")

    # --- Paso 2: aplicar muestra si se indica ---
    if sample_size is not None:
        n = min(sample_size, len(df_con_mbid))
        df_muestra = df_con_mbid.sample(n=n, random_state=42).reset_index(drop=True)
        log.info(f"  Muestra seleccionada       : {len(df_muestra):,}")
    else:
        df_muestra = df_con_mbid.reset_index(drop=True)
        log.info(f"  Procesando todos los tracks con MBID")
    log.info("=" * 55)

    # --- Paso 3: iterar sobre cada MBID y llamar a la API ---
    resultados  = []
    ok_ll = ok_hl = fail_ll = fail_hl = 0
    total = len(df_muestra)
    inicio = time.time()

    for i, fila in enumerate(df_muestra.itertuples(index=False), start=1):
        mbid = fila.mbid

        # Llamada 1: low-level (bpm, key, scale)
        ll = fetch_low_level(mbid)
        time.sleep(SLEEP_SECONDS)

        # Llamada 2: high-level (danceability, mood_happy, genre_ab)
        hl = fetch_high_level(mbid)
        time.sleep(SLEEP_SECONDS)

        # Contar éxitos y fallos
        if ll["bpm"] is not None:
            ok_ll += 1
        else:
            fail_ll += 1

        if hl["genre_ab"] is not None:
            ok_hl += 1
        else:
            fail_hl += 1

        # Guardar resultado de este track
        resultados.append({"mbid": mbid, **ll, **hl})

        # Mostrar progreso cada LOG_EVERY_N tracks
        if i % LOG_EVERY_N == 0 or i == total:
            tiempo_pasado = time.time() - inicio
            velocidad     = (i * 2) / tiempo_pasado if tiempo_pasado > 0 else 0
            eta_minutos   = ((total - i) * 2) / (velocidad * 60) if velocidad > 0 else 0
            log.info(
                f"  [{i:>5}/{total}] "
                f"Low ✓{ok_ll} ✗{fail_ll} | "
                f"High ✓{ok_hl} ✗{fail_hl} | "
                f"ETA: {eta_minutos:.1f} min"
            )

        # Guardar checkpoint parcial cada CHECKPOINT_EVERY_N tracks
        if i % CHECKPOINT_EVERY_N == 0:
            checkpoint = pd.DataFrame(resultados)
            ruta_ckpt  = os.path.join(OUTPUT_DIR, f"checkpoint_{i}_{timestamp}.csv")
            checkpoint.to_csv(ruta_ckpt, index=False)
            log.info(f"  💾 Checkpoint guardado: {ruta_ckpt}")

    # --- Paso 4: construir DataFrame de features y hacer merge ---

    # DataFrame con los datos de audio obtenidos
    df_features = pd.DataFrame(resultados)
    df_features = df_features.drop_duplicates(subset="mbid")  # evitar duplicados

    # Left merge por mbid: conserva TODOS los tracks de la muestra
    # Los que no tienen datos en AcousticBrainz quedan con NaN → correcto
    df_enriquecido = df_muestra.merge(df_features, on="mbid", how="left")

    # --- Paso 5: seleccionar las 11 columnas objetivo ---
    columnas_finales = [
        "name", "artist",
        "bpm", "key", "scale",
        "danceability", "mood_happy", "genre_ab",
        "country", "playcount", "listeners"
    ]
    df_final = df_enriquecido[columnas_finales].copy()

    # --- Guardar resultado final ---
    ruta_salida = os.path.join(OUTPUT_DIR, f"lastfm_enriched_{timestamp}.csv")
    df_final.to_csv(ruta_salida, index=False)

    # --- Mostrar resumen ---
    log.info("\n" + "=" * 55)
    log.info("  RESUMEN FINAL")
    log.info("=" * 55)
    log.info(f"  Tracks procesados    : {total:,}")
    log.info(f"  Low-level  OK        : {ok_ll:,} ({ok_ll/total*100:.1f}%)")
    log.info(f"  High-level OK        : {ok_hl:,} ({ok_hl/total*100:.1f}%)")
    log.info(f"  Low-level  sin datos : {fail_ll:,} ({fail_ll/total*100:.1f}%)")
    log.info(f"  High-level sin datos : {fail_hl:,} ({fail_hl/total*100:.1f}%)")
    log.info(f"  Archivo guardado en  : {ruta_salida}")
    log.info("=" * 55)

    return df_final


# =============================================================================
#  EJECUCIÓN PRINCIPAL
# =============================================================================

if __name__ == "__main__":

    # 1. Cargar el dataset
    log.info(f"Cargando dataset: {CSV_PATH}")
    df_clean = pd.read_csv(CSV_PATH)
    log.info(f"Shape: {df_clean.shape} | MBIDs válidos: {df_clean['mbid'].notna().sum():,}")

    # 2. Ejecutar el pipeline
    #    Elige la opción que necesites:

    # OPCIÓN A — muestra de 2.000 tracks (~45-60 min) ← empieza aquí
    df_resultado = enrich_dataset_with_acousticbrainz(df_clean, sample_size=2000)

    # OPCIÓN B — muestra de 10.000 tracks (~5-6 horas)
    # df_resultado = enrich_dataset_with_acousticbrainz(df_clean, sample_size=10000)

    # OPCIÓN C — todos los tracks con MBID (~15-20 horas)
    # df_resultado = enrich_dataset_with_acousticbrainz(df_clean, sample_size=None)

    # 3. Inspección rápida del resultado
    print("\n── Primeras filas ──")
    print(df_resultado.head(10).to_string(index=False))

    print("\n── Cobertura de columnas de audio ──")
    for col in ["bpm", "key", "scale", "danceability", "mood_happy", "genre_ab"]:
        n   = df_resultado[col].notna().sum()
        pct = n / len(df_resultado) * 100
        barra = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
        print(f"  {col:<15} {barra}  {n:>5}/{len(df_resultado):,}  ({pct:.1f}%)")

    print("\n── Estadísticas básicas ──")
    print(df_resultado[["bpm", "danceability", "mood_happy"]].describe().round(3))

15:41:49  INFO      Cargando dataset: src/lastfm_dataset.csv


15:41:50  INFO      Shape: (59999, 10) | MBIDs válidos: 53,294
15:41:50  INFO      =======================================================
15:41:50  INFO        INICIO DEL PIPELINE DE ENRIQUECIMIENTO
15:41:50  INFO      =======================================================
15:41:50  INFO        Total tracks en el dataset : 59,999
15:41:50  INFO        Tracks con MBID válido     : 53,294
15:41:50  INFO        Tracks sin MBID (→ NaN)    : 6,705
15:41:50  INFO        Muestra seleccionada       : 2,000
15:41:50  INFO      =======================================================
15:43:28  INFO        [  100/2000] Low ✓8 ✗92 | High ✓8 ✗92 | ETA: 31.1 min
15:45:09  INFO        [  200/2000] Low ✓13 ✗187 | High ✓13 ✗187 | ETA: 29.9 min
15:46:32  INFO        [  300/2000] Low ✓16 ✗284 | High ✓16 ✗284 | ETA: 26.7 min
15:47:55  INFO        [  400/2000] Low ✓21 ✗379 | High ✓21 ✗379 | ETA: 24.3 min
15:49:17  INFO        [  500/2000] Low ✓28 ✗472 | High ✓28 ✗472 | ETA: 22.4 min
15:49:17  INFO        

KeyboardInterrupt: 

In [6]:
df_sample_metadata = pd.read_csv('./checkpoint_500_20260327_1541.csv')
df_sample_metadata

,mbid,bpm,key,scale,danceability,mood_happy,genre_ab
0,042c0c6b-0e01-423f-adb2-14bf0948e3d4,NaN,NaN,NaN,NaN,NaN,NaN
1,0379ed24-fd11-4dd2-b884-c21a14f22238,NaN,NaN,NaN,NaN,NaN,NaN
2,05f863d9-0c54-4d9e-b7e2-9e0d1241dad9,NaN,NaN,NaN,NaN,NaN,NaN
3,06c6adce-6596-4798-831a-01fd84bc38b1,NaN,NaN,NaN,NaN,NaN,NaN
4,296fbbd4-57b1-41f6-8394-da2ea3bb9c10,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
495,090e54b3-0534-44eb-a505-91a8c8b13f23,NaN,NaN,NaN,NaN,NaN,NaN
496,0b4ad065-0aeb-3454-97f8-ea9a735292d7,NaN,NaN,NaN,NaN,NaN,NaN
497,67e38898-9200-4f2a-bc99-3f225dedbf4a,NaN,NaN,NaN,NaN,NaN,NaN
498,095b6855-8eba-4ee1-ab57-aa26c65d5a55,104.719948,C#,major,0.900248,0.5,folkcountry


In [10]:
df_sample_metadata.isnull().sum()

mbid              0
bpm             472
key             472
scale           472
danceability    472
mood_happy      472
genre_ab        472
dtype: int64

**Conclusiones de los problemas al descargar datos hasta el momento:**

**· Problema 1:** Descarga de datos con API Lastfm: 
* Inicialmente se utilizan los endpoints "chart.getTopTracks","geo.getTopTracks", "tag.getTopTracks" y "track.getInfo".
* Se han hecho varias pruebas de descarga de datos con el objetivo de análisis de los tracks en el mercado musical según streams, popularidad y escuchas pero se ha encontrado data insuficiente con muchos valores faltantes, con respuestas muy generales o poco coherentes para el análisis.

**Resolución de la descarga de datos:** Búsqueda de nuevas apis para enfocar mejor el análisis y obtener más data. Se decide utilizar Acousticbrainz porque contiene la metadata de cada track. 

**· Problema 2:** Descarga de datos con Acousticbrainz:
* Se intenta recolectar data de los endpoints "Low-level " y "High-level" que devuelven la información del bpm, key, scale,danceability, mood_happy y genre_ab.
* Al analizar una muestra de 500 datos se oberva que existen nulos en todas las columnas menos en mbid. Esto indica que de las canciones que tenemos en el csv de Lastfm no hay data. Es posible que sea debido a que la misma base de datos dónde buscamos la metadata se dejó de actualizar antes que Lastfm."

**Resolución de** Ideas, (1) utilizar solamente api acousticbrainz con data antigua com prueba, (2) intentar hacer web scrapping se paginas como Nina Protocol y BAndcamp. 


# Nina Protocol y Bandcamp

## Nina protocol

APIS complementarias:

* Last.fm → mainstream

* Nina → underground

In [11]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

URL = "https://www.ninaprotocol.com/explore"

headers = {
    "User-Agent": "Mozilla/5.0"
}

def scrape_nina_explore():
    response = requests.get(URL, headers=headers)

    if response.status_code != 200:
        print("Error al acceder a Nina")
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    releases = []

    # ⚠️ Esto puede cambiar si Nina cambia HTML
    cards = soup.find_all("a")

    for card in cards:
        text = card.get_text(strip=True)

        if text and len(text) < 100:
            link = card.get("href")

            if link and "/release/" in link:
                releases.append({
                    "title_artist": text,
                    "link": "https://www.ninaprotocol.com" + link
                })

    return pd.DataFrame(releases)

ModuleNotFoundError: No module named 'bs4'

In [ ]:
df_nina = scrape_nina_explore()

df_nina.head()

In [ ]:
df_nina[['title', 'artist']] = df_nina['title_artist'].str.split(' - ', n=1, expand=True)

In [ ]:
time.sleep(1)

In [ ]:
df_nina['artist'].value_counts().head(10)